# 09 — PA-only log-Mel baseline

Trains one log-Mel replay CNN on ASVspoof 2019 PA only, then scores it on the official PA evaluation list and the official 2017 evaluation list. Minimum t-DCF is computed on the PA predictions.

The checkpoint is written to `inverted_mel_pa2019_train/runs/log_mel_pa2019/`. That is not the existing inverted-Mel folder. The saved file is still named `best_inverted_mel_pa2019.pt`; the front-end stored inside it is log-Mel.

Training is about an hour. Do not start another GPU job while this notebook is running. The PA evaluation list is 134,730 files, so scoring is the long step after training.

When the last cell prints the row, add it to the paper as PA-only log-Mel. Do not type a number in before that print.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "common.py").exists():
    HERE = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\paper_official_eval")
sys.path.insert(0, str(HERE))

from common import asv17_root, ensure_pa_ready, experiments_dir, pa_root, paper_runs, run

ensure_pa_ready()
EXP = experiments_dir()
PY = sys.executable
TRAIN_DIR = EXP / "inverted_mel_pa2019_train"
OUT = TRAIN_DIR / "runs" / "log_mel_pa2019"
PROTECTED = (TRAIN_DIR / "runs" / "inverted_mel_pa2019").resolve()
if OUT.resolve() == PROTECTED:
    raise RuntimeError("Refusing to write into the existing inverted-Mel PA run.")
CKPT = OUT / "best_inverted_mel_pa2019.pt"

run([
    PY, str(TRAIN_DIR / "train_pa2019.py"),
    "--pa-root", str(pa_root()),
    "--feature-type", "mel",
    "--output", str(OUT),
    "--epochs", "15",
    "--patience", "4",
    "--batch-size", "8",
])
if not CKPT.is_file():
    raise FileNotFoundError(CKPT)
print("checkpoint", CKPT)

In [ ]:
PA_OUT = paper_runs() / "pa_only_logmel_on_pa_eval"
ASV_OUT = paper_runs() / "pa_only_logmel_on_2017_eval"

run([
    PY, str(EXP / "inverted_mel_pa2019_full" / "eval_full_pa2019.py"),
    "--pa-root", str(pa_root()),
    "--checkpoint", str(CKPT),
    "--split", "eval",
    "--batch-size", "16",
    "--output", str(PA_OUT),
])
run([
    PY, str(EXP / "inverted_mel_pa2019_on_asvspoof2017" / "eval_asvspoof2017.py"),
    "--data-root", str(asv17_root()),
    "--checkpoint", str(CKPT),
    "--split", "eval",
    "--batch-size", "16",
    "--output", str(ASV_OUT),
])

In [ ]:
import json

from compute_pa_tdcf import min_tdcf_for_predictions

pa_metrics = json.loads((PA_OUT / "pa2019_eval_full_metrics.json").read_text(encoding="utf-8"))
asv_metrics = json.loads((ASV_OUT / "asvspoof2017_eval_metrics.json").read_text(encoding="utf-8"))
pa_eer = pa_metrics["metrics_at_oracle_eer_threshold"]["eer_percent"]
asv_eer = asv_metrics["metrics_at_oracle_eer_threshold"]["eer_percent"]
tdcf = min_tdcf_for_predictions(PA_OUT / "pa2019_eval_full_predictions.csv")
if pa_metrics["feature_type"] != "mel":
    raise RuntimeError(f"Expected log-Mel, found {pa_metrics['feature_type']}")

row = {
    "model": "PA-only log-Mel",
    "feature_type": pa_metrics["feature_type"],
    "checkpoint": str(CKPT),
    "eer_2017": asv_eer,
    "n_2017": asv_metrics["num_scored_files"],
    "eer_pa": pa_eer,
    "n_pa": pa_metrics["num_scored_files"],
    "n_pa_protocol": pa_metrics["num_protocol_files"],
    "skipped_pa": pa_metrics["num_skipped_corrupt"],
    "min_tdcf": tdcf,
}
summary_path = paper_runs() / "pa_only_logmel_eval.json"
summary_path.write_text(json.dumps(row, indent=2), encoding="utf-8")
print(
    f"PA-only log-Mel | 2017 {asv_eer:.2f}% | PA {pa_eer:.2f}% | t-DCF {tdcf:.4f}"
)
print("scored", row["n_2017"], "of 2017 and", row["n_pa"], "of", row["n_pa_protocol"], "PA")
print("wrote", summary_path)